In [1]:
!source myenv/bin/activate

In [2]:
%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached https://download.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp310-cp310-linux_x86_64.whl.metadata (28 kB)
  Using cached https://download.pytorch.org/whl/cu124/torchvision-0.21.0%2Bcu124-cp310-cp310-linux_x86_64.whl.metadata (6.1 kB)
Using cached https://download.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp310-cp310-linux_x86_64.whl (768.4 MB)
Using cached https://download.pytorch.org/whl/cu124/torchvision-0.21.0%2Bcu124-cp310-cp310-linux_x86_64.whl (7.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchvision] [torchvision]
ERROR: pip's dependency resolver does not currently t

In [3]:
!pip install -q tokenizers matplotlib seaborn accelerate opencv-python numpy pillow accelerate bitsandbytes shapely wandb dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.2 requires transformers<5.0.0,>=4.41.0, which is not installed.

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Uninstall existing conflicting packages first
%pip uninstall -y numpy transformers

# Install compatible numpy and bleeding edge transformers
%pip install "numpy<2.0"  # Transformers often has issues with Numpy 2.0+
%pip install git+https://github.com/huggingface/transformers.git
%pip install --upgrade qwen-vl-utils

In [ ]:
import os
from IPython.display import display
import json
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import torch
import cv2
import numpy as np
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import re
from eval import GeoNLIEvaluator
# --- 1. MODEL SETUP ---
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Qwen3-VL on {device}...")
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype="auto", device_map="auto"
)
qwen_processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
"""
SAM3-VLM Router Pipeline for Remote Sensing VQA Tasks
Routes questions to either SAM3 (segmentation-required) or VLM (direct inference)
"""

import os
import json
from pathlib import Path
from PIL import Image
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from tqdm import tqdm
import glob

# --- 1. MODEL SETUP ---
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Qwen3-VL on {device}...")
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype="auto", device_map="auto"
)
qwen_processor = AutoProcessor.from_pretrained(MODEL_ID)

# --- 2. ROUTER SYSTEM PROMPT ---
ROUTER_SYSTEM_PROMPT = """You are an intelligent routing system for remote sensing visual question answering tasks. Your job is to analyze questions and determine whether they require SAM3 (Segment Anything Model 3) segmentation capabilities or can be answered directly by a Vision Language Model (VLM).

## SAM3 Capabilities
SAM3 is a foundation model for promptable segmentation that can:
- Detect and segment ALL instances of objects specified by text prompts (e.g., "buildings", "trees", "vehicles")
- Generate precise pixel-level segmentation masks for multiple object instances
- Provide bounding boxes and confidence scores for detected objects
- Segment objects exhaustively across the entire image
- Handle open-vocabulary concepts (270K+ unique concepts)
- Return masks that enable precise geometric calculations (area, perimeter, length, orientation)

## When to Route to SAM3
Route to SAM3 when the question requires:

1. **Counting/Quantification**: Questions asking "how many", "number of", "count", "amount of" specific objects
   - Examples: "How many buildings are in the image?", "What is the number of cars?"

2. **Area/Coverage Calculations**: Questions about area covered, percentage of coverage, spatial extent
   - Examples: "What is the area covered by buildings?", "What percentage of land is forested?"

3. **Length/Distance Measurements**: Questions about length, width, perimeter, or distance
   - Examples: "What is the length of the road?", "What is the perimeter of the lake?"

4. **Orientation/Angle Analysis**: Questions about direction, orientation, angle, or alignment
   - Examples: "What is the orientation of the runway?", "What direction is the road heading?"

5. **Density/Concentration**: Questions about density, distribution, or concentration of objects
   - Examples: "What is the density of trees?", "How concentrated are the buildings?"

6. **Ratio/Proportion Comparisons**: Questions comparing quantities or areas between different object types
   - Examples: "Are there more residential buildings than commercial buildings?", "What is the ratio of water to land?"

7. **Spatial Relationships Requiring Segmentation**: Questions about adjacency, overlap, or precise spatial arrangements
   - Examples: "Which buildings are adjacent to the road?", "What percentage of the park is covered by trees?"

8. **Size/Dimension Analysis**: Questions about the size, dimensions, or scale of objects
   - Examples: "What is the size of the largest building?", "How wide is the river?"

## When to Route to VLM
Route to VLM when the question can be answered through visual understanding alone:

1. **Object Existence/Presence**: Binary questions about whether something exists
   - Examples: "Is a residential building present?", "Are there any cars in the image?"

2. **Visual Attributes**: Questions about color, texture, appearance, or visual characteristics
   - Examples: "What color is the roof?", "What is the texture of the ground?"

3. **Object Recognition/Classification**: Questions about identifying or classifying visible objects
   - Examples: "What type of building is this?", "Is this an urban or rural area?"

4. **Scene Understanding**: Questions about overall scene characteristics, context, or environment
   - Examples: "What is the weather condition?", "What time of day is it?", "Is this a commercial area?"

5. **Qualitative Descriptions**: Questions asking for descriptions without precise measurements
   - Examples: "Describe the landscape", "What is the condition of the road?"

6. **Approximate Comparisons**: Questions using "more than", "less than" without requiring exact counts
   - Examples: "Are there more trees than buildings?" (can be visually estimated)

7. **Pattern Recognition**: Questions about patterns, arrangements, or general layouts
   - Examples: "What is the pattern of streets?", "How are the buildings arranged?"

## Critical Routing Decision Guidelines
- If ANY geometric measurement is required (area, length, count, density, ratio), route to SAM3
- If the question uses quantitative terms ("how many", "what is the area", "count", "number"), route to SAM3
- If the question requires distinguishing between multiple instances of the same object type, route to SAM3
- If the question can be answered with visual observation alone, route to VLM
- When in doubt between SAM3 and VLM, prefer SAM3 for quantitative questions and VLM for qualitative questions

## Output Format
Respond with ONLY a single word:
- "SAM" - if the question requires SAM3 segmentation
- "VLM" - if the question can be answered by VLM alone

Do not provide any explanation, reasoning, or additional text. Only output "SAM" or "VLM".

## Examples

Question: "What is the amount of residential buildings in the image?"
Answer: SAM

Question: "What is the area covered by buildings?"
Answer: SAM

Question: "What is the number of buildings?"
Answer: SAM

Question: "Are there more residential buildings than roads?"
Answer: SAM

Question: "Is a residential building present?"
Answer: VLM

Question: "What color is the largest building?"
Answer: VLM

Question: "What is the density of vegetation in the area?"
Answer: SAM

Question: "What type of terrain is visible?"
Answer: VLM

Question: "How many cars are parked in the lot?"
Answer: SAM

Question: "Is this an urban or rural setting?"
Answer: VLM

Question: "What is the length of the bridge?"
Answer: SAM

Question: "What is the orientation of the runway?"
Answer: SAM

Question: "Are there trees in the image?"
Answer: VLM

Question: "What is the ratio of green space to built area?"
Answer: SAM

Question: "Describe the architectural style of the buildings."
Answer: VLM"""


# --- 3. ROUTING FUNCTION ---
def route_question(question: str) -> str:
    """
    Routes a question to either SAM3 or VLM based on the routing system prompt.
    
    Args:
        question: The VQA question to route
        
    Returns:
        "SAM" or "VLM" indicating which system should handle the question
    """
    messages = [
        {
            "role": "system",
            "content": ROUTER_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Question: {question}\nAnswer:"
        }
    ]
    
    # Prepare the text for the model
    text = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    # Process without image (routing is text-only decision)
    inputs = qwen_processor(
        text=[text],
        padding=True,
        return_tensors="pt",
    ).to(device)
    
    # Generate response
    with torch.no_grad():
        generated_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=10,  # Only need "SAM" or "VLM"
            temperature=0.1,    # Low temperature for consistent routing
            do_sample=False,    # Deterministic routing
        )
    
    # Extract the generated text
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    output_text = qwen_processor.batch_decode(
        generated_ids_trimmed, 
        skip_special_tokens=True, 
        clean_up_tokenization_spaces=False
    )[0]
    
    # Clean and validate output
    route = output_text.strip().upper()
    
    # Handle variations and ensure valid output
    if "SAM" in route:
        return "SAM"
    elif "VLM" in route:
        return "VLM"
    else:
        # Default to VLM if unclear (conservative choice)
        print(f"Warning: Unclear routing decision '{route}' for question: {question[:50]}...")
        return "VLM"


# --- 4. MAIN PIPELINE FOR VRSBench ---
def process_vrsbench_dataset(
    annotations_dir: str,
    output_sam_json: str,
    output_vlm_json: str,
    max_files: int = 700
):
    """
    Process VRSBench dataset JSON files and route questions to SAM3 or VLM.
    
    Args:
        annotations_dir: Path to VRSBench_val/Annotations_val/ directory
        output_sam_json: Path to save SAM-routed questions
        output_vlm_json: Path to save VLM-routed questions
        max_files: Maximum number of JSON files to process
    """
    print(f"\n{'='*80}")
    print(f"SAM3-VLM Router Pipeline - VRSBench Dataset")
    print(f"{'='*80}\n")
    
    # Find all JSON files
    json_pattern = os.path.join(annotations_dir, "*.json")
    json_files = sorted(glob.glob(json_pattern))[:max_files]
    
    print(f"Found {len(json_files)} JSON files to process")
    print(f"Processing directory: {annotations_dir}\n")
    
    # Storage for results
    sam_results = []
    vlm_results = []
    sam_count = 0
    vlm_count = 0
    total_questions = 0
    
    # Track question types
    question_type_stats = {}
    route_by_type = {"SAM": {}, "VLM": {}}
    
    # Process each JSON file
    print("Routing questions from JSON files...")
    for json_file in tqdm(json_files, desc="Processing files"):
        try:
            with open(json_file, 'r') as f:
                data = json.load(f)
            
            # Extract metadata
            image_name = data.get("image", "")
            caption = data.get("caption", "")
            objects = data.get("objects", [])
            qa_pairs = data.get("qa_pairs", [])
            
            # Process each QA pair
            for qa_pair in qa_pairs:
                ques_id = qa_pair.get("ques_id")
                question = qa_pair.get("question", "")
                answer = qa_pair.get("answer", "")
                q_type = qa_pair.get("type", "unknown")
                
                if not question:
                    continue
                
                total_questions += 1
                
                # Track question type statistics
                if q_type not in question_type_stats:
                    question_type_stats[q_type] = 0
                question_type_stats[q_type] += 1
                
                # Route the question
                route = route_question(question)
                
                # Create result entry
                result_entry = {
                    "file": os.path.basename(json_file),
                    "image": image_name,
                    "ques_id": ques_id,
                    "question": question,
                    "answer": answer,
                    "question_type": q_type,
                    "caption": caption,
                    "objects": objects,
                    "route_to": route
                }
                
                # Add to appropriate list
                if route == "SAM":
                    sam_results.append(result_entry)
                    sam_count += 1
                    if q_type not in route_by_type["SAM"]:
                        route_by_type["SAM"][q_type] = 0
                    route_by_type["SAM"][q_type] += 1
                else:
                    vlm_results.append(result_entry)
                    vlm_count += 1
                    if q_type not in route_by_type["VLM"]:
                        route_by_type["VLM"][q_type] = 0
                    route_by_type["VLM"][q_type] += 1
                    
        except Exception as e:
            print(f"\nError processing {json_file}: {str(e)}")
            continue
    
    # Calculate statistics
    sam_percentage = round(sam_count / total_questions * 100, 2) if total_questions > 0 else 0
    vlm_percentage = round(vlm_count / total_questions * 100, 2) if total_questions > 0 else 0
    
    # Prepare SAM output
    sam_output = {
        "metadata": {
            "dataset": "VRSBench",
            "total_files_processed": len(json_files),
            "total_questions_routed_to_sam": sam_count,
            "percentage_of_total": sam_percentage,
            "question_types": route_by_type["SAM"]
        },
        "questions": sam_results
    }
    
    # Prepare VLM output
    vlm_output = {
        "metadata": {
            "dataset": "VRSBench",
            "total_files_processed": len(json_files),
            "total_questions_routed_to_vlm": vlm_count,
            "percentage_of_total": vlm_percentage,
            "question_types": route_by_type["VLM"]
        },
        "questions": vlm_results
    }
    
    # Print summary
    print(f"\n{'='*80}")
    print(f"Routing Summary - VRSBench Dataset")
    print(f"{'='*80}")
    print(f"Total JSON Files:    {len(json_files)}")
    print(f"Total Questions:     {total_questions}")
    print(f"Routed to SAM3:      {sam_count} ({sam_percentage}%)")
    print(f"Routed to VLM:       {vlm_count} ({vlm_percentage}%)")
    print(f"\nQuestion Type Distribution:")
    print(f"-" * 80)
    for q_type, count in sorted(question_type_stats.items(), key=lambda x: x[1], reverse=True):
        sam_type_count = route_by_type["SAM"].get(q_type, 0)
        vlm_type_count = route_by_type["VLM"].get(q_type, 0)
        print(f"  {q_type:30s}: {count:4d} total (SAM: {sam_type_count:4d}, VLM: {vlm_type_count:4d})")
    print(f"{'='*80}\n")
    
    # Save to files
    print(f"Saving SAM results to: {output_sam_json}")
    with open(output_sam_json, 'w') as f:
        json.dump(sam_output, f, indent=2)
    
    print(f"Saving VLM results to: {output_vlm_json}")
    with open(output_vlm_json, 'w') as f:
        json.dump(vlm_output, f, indent=2)
    
    print(f"\n✓ Routing complete!\n")
    
    # Display sample results
    print("Sample SAM Routing Results:")
    print("-" * 80)
    for result in sam_results[:5]:
        print(f"🎯 SAM | [{result['question_type']}] {result['question'][:60]}")
    print("-" * 80)
    
    print("\nSample VLM Routing Results:")
    print("-" * 80)
    for result in vlm_results[:5]:
        print(f"👁️  VLM | [{result['question_type']}] {result['question'][:60]}")
    print("-" * 80)


# --- 5. USAGE FUNCTION ---
def get_routing_decision(question: str) -> str:
    """
    Convenience function to get routing decision for a single question.
    
    Args:
        question: The VQA question to route
        
    Returns:
        "SAM" or "VLM"
    """
    return route_question(question)


# --- 6. MAIN EXECUTION ---
if __name__ == "__main__":
    # Configuration for VRSBench
    ANNOTATIONS_DIR = "VRSBench_val/Annotations_val"
    OUTPUT_SAM_JSON = "vrsbench_routed_to_sam.json"
    OUTPUT_VLM_JSON = "vrsbench_routed_to_vlm.json"
    MAX_FILES = 700
    
    # Run the pipeline
    process_vrsbench_dataset(
        annotations_dir=ANNOTATIONS_DIR,
        output_sam_json=OUTPUT_SAM_JSON,
        output_vlm_json=OUTPUT_VLM_JSON,
        max_files=MAX_FILES
    )
    
    print("\nExample usage for single question:")
    print("-" * 80)
    test_question = "What is the prominent man-made structure in the image?"
    route = get_routing_decision(test_question)
    print(f"Question: {test_question}")
    print(f"Route to: {route}")
    print("-" * 80)